# CASE 07 — Square 8×8 + 5×5 — Convolution TopLeft

In [1]:
%%writefile case07_sq_5x5_convTopLeft.cu

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define WIDTH       8
#define HEIGHT      8
#define MASK_WIDTH  5
#define MASK_HEIGHT 5
#define BLOCK_SIZE  8

__global__ void convolutionTopLeft(int *dA, int *dMask, int *dC,
                                   int width, int height,
                                   int mWidth, int mHeight)
{
    int col = threadIdx.x + blockIdx.x * blockDim.x;
    int row = threadIdx.y + blockIdx.y * blockDim.y;
    if (row < height && col < width)
    {
        int sum = 0;
        for (int i = 0; i < mHeight; i++)
            for (int j = 0; j < mWidth; j++)
            {
                int r = row + i;        /* top-left */
                int c = col + j;
                if (r < height && c < width)
                    sum += dA[r*width+c] *
                           dMask[(mHeight-1-i)*mWidth+(mWidth-1-j)]; /* flipped */
            }
        dC[row*width+col] = sum;
    }
}

void printMatrix(const char *label, int *M, int w, int h)
{
    printf("\n%s:\n", label);
    for (int r = 0; r < h; r++)
    {
        for (int c = 0; c < w; c++)
            printf("%6d", M[r*w+c]);
        printf("\n");
    }
}

int main()
{
    int size     = WIDTH * HEIGHT * sizeof(int);
    int maskSize = MASK_WIDTH * MASK_HEIGHT * sizeof(int);

    int *hA    = (int*) malloc(size);
    int *hMask = (int*) malloc(maskSize);
    int *hC    = (int*) malloc(size);

    srand(time(NULL));
    for (int i = 0; i < WIDTH*HEIGHT; i++)
        hA[i] = rand()%9+1;

    int tempMask[5][5] = {
        {1,1,1,1,1},
        {1,2,2,2,1},
        {1,2,4,2,1},
        {1,2,2,2,1},
        {1,1,1,1,1}
    };
    for (int i = 0; i < MASK_HEIGHT; i++)
        for (int j = 0; j < MASK_WIDTH; j++)
            hMask[i*MASK_WIDTH+j] = tempMask[i][j];

    printMatrix("Input Matrix (8x8)", hA,    WIDTH,      HEIGHT);
    printMatrix("Mask (5x5)",         hMask, MASK_WIDTH, MASK_HEIGHT);

    int *dA, *dMask, *dC;
    cudaMalloc((void**)&dA,    size);
    cudaMalloc((void**)&dMask, maskSize);
    cudaMalloc((void**)&dC,    size);
    cudaMemcpy(dA,    hA,    size,     cudaMemcpyHostToDevice);
    cudaMemcpy(dMask, hMask, maskSize, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, BLOCK_SIZE, 1);
    dim3 DimGrid((int)ceil((float)WIDTH/BLOCK_SIZE),
                 (int)ceil((float)HEIGHT/BLOCK_SIZE), 1);

    cudaEvent_t start, stop; float gpuTime;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);

    convolutionTopLeft<<<DimGrid,DimBlock>>>(dA,dMask,dC,
                        WIDTH,HEIGHT,MASK_WIDTH,MASK_HEIGHT);

    cudaEventRecord(stop); cudaEventSynchronize(stop);
    cudaEventElapsedTime(&gpuTime, start, stop);
    cudaMemcpy(hC, dC, size, cudaMemcpyDeviceToHost);

    printMatrix("OUTPUT: Convolution TopLeft (8x8, 5x5)", hC, WIDTH, HEIGHT);
    printf("\nGPU Time: %.4f ms\n", gpuTime);
    printf("Grid: %dx%d  Block: %dx%d\n",
            DimGrid.x,DimGrid.y,DimBlock.x,DimBlock.y);

    cudaFree(dA); cudaFree(dMask); cudaFree(dC);
    free(hA); free(hMask); free(hC);
    cudaEventDestroy(start); cudaEventDestroy(stop);
    return 0;
}

Writing case07_sq_5x5_convTopLeft.cu


In [4]:
!nvcc -arch=sm_75 case07_sq_5x5_convTopLeft.cu -o case07_sq_5x5_convTopLeft

!./case07_sq_5x5_convTopLeft


Input Matrix (8x8):
     6     9     2     2     7     5     8     3
     3     8     3     8     3     9     6     9
     4     1     5     4     5     6     4     4
     4     8     1     4     2     1     9     8
     9     2     7     5     4     5     7     6
     1     7     4     3     4     9     9     7
     7     4     2     2     8     5     3     2
     2     2     5     3     9     3     8     8

Mask (5x5):
     1     1     1     1     1
     1     2     2     2     1
     1     2     4     2     1
     1     2     2     2     1
     1     1     1     1     1

OUTPUT: Convolution TopLeft (8x8, 5x5):
   168   159   174   191   175   138    85    30
   148   163   167   186   185   153    87    34
   162   152   163   186   183   143    80    27
   153   154   175   206   188   138    82    31
   126   137   167   178   157   113    67    23
    91   105   125   125   120    91    47    17
    54    60    63    71    65    45    29    10
    21    22    28    31    28    1